In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk
import pandas as pd
from tqdm import tqdm


In [2]:
model_id = "unb-labia/BERTomelo-ModernBERT-Base-v1"
#model_id = "unb-labia/BERTomelo-ModernBERT-Large-STS-8k"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# LENER

In [39]:
lener = load_from_disk('../lener_br_local')

In [ ]:
qtds_lener = []
for k in lener.keys():
    for item in lener[k]['tokens']:
        qtds_lener.append({"palavras": len(item), "tokens": tokenizer(item, is_split_into_words=True, return_tensors="pt")['input_ids'].shape[1]})


In [76]:
df_lener = pd.DataFrame(qtds_lener)
df_lener

,palavras,tokens
0,47,70
1,55,78
2,25,38
3,55,79
4,10,17
...,...,...
10390,88,137
10391,8,13
10392,17,150
10393,3,29


In [77]:
print('Describe:', df_lener['palavras'].describe())


Describe: count    10395.000000
mean        30.598653
std         35.513078
min          0.000000
25%          6.000000
50%         22.000000
75%         43.000000
max        755.000000
Name: palavras, dtype: float64


In [78]:
print('Describe:', df_lener['tokens'].describe())

Describe: count    10395.000000
mean        54.556614
std         62.554015
min          2.000000
25%         12.000000
50%         38.000000
75%         74.000000
max       1290.000000
Name: tokens, dtype: float64


# ASSIN2

In [67]:
ds_assin2 = load_dataset('nilc-nlp/assin2')
ds_assin2

DatasetDict({
    train: Dataset({
        features: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment'],
        num_rows: 6500
    })
    test: Dataset({
        features: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment'],
        num_rows: 2448
    })
    validation: Dataset({
        features: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment'],
        num_rows: 500
    })
})

In [89]:
qtds_assin2 = []

for k in ds_assin2.keys():
    for item in ds_assin2[k]:
        qtds_assin2.append({"palavras": len(item["premise"].split()) + len(item["hypothesis"].split()), "tokens": tokenizer(item["premise"], item["hypothesis"], return_tensors="pt")['input_ids'].shape[1]})

In [90]:
df_assin2 = pd.DataFrame(qtds_assin2)
df_assin2

,palavras,tokens
0,22,29
1,23,39
2,18,36
3,13,21
4,14,18
...,...,...
9443,12,17
9444,16,25
9445,23,32
9446,13,18


In [91]:
print('Describe:', df_assin2['palavras'].describe())

Describe: count    9448.000000
mean       18.470576
std         6.748329
min         7.000000
25%        13.000000
50%        17.000000
75%        22.000000
max        59.000000
Name: palavras, dtype: float64


In [92]:
print('Describe:', df_assin2['tokens'].describe())

Describe: count    9448.000000
mean       26.654953
std         9.253365
min        10.000000
25%        20.000000
50%        25.000000
75%        32.000000
max        81.000000
Name: tokens, dtype: float64


# mMARCO

In [12]:
dataset = load_dataset('unicamp-dl/mmarco', 'portuguese', split="train", trust_remote_code=True, keep_in_memory=False)
ds_len = len(dataset)
#del dataset
#dataset = load_dataset('unicamp-dl/mmarco', 'portuguese', split="train", trust_remote_code=True, streaming=True)


Repo card metadata block was not found. Setting CardData to empty.


Loading dataset shards:   0%|          | 0/67 [00:00<?, ?it/s]

In [13]:
dataset

Dataset({
    features: ['query', 'positive', 'negative'],
    num_rows: 39780811
})

In [28]:
def count_tokens(batch):
    qtds_mmarco = []
    for item in range(len(batch['query'])):
        qtds_mmarco.append(pd.Series([{"palavras_query": len(batch["query"][item].split()), "palavras_positive": len(batch["positive"][item].split()), "palavras_negative": len(batch["negative"][item].split()), 
                                       "qtd_palavras_query": len(batch["query"][item].split()) + len(batch["positive"][item].split()) + len(batch["negative"][item].split()),
                                "tokens_query": tokenizer(batch["query"][item], return_tensors="pt")['input_ids'].shape[1], "tokens_positive": tokenizer(batch["positive"][item], return_tensors="pt")['input_ids'].shape[1], "tokens_negative": tokenizer(batch["negative"][item], return_tensors="pt")['input_ids'].shape[1],
                                "qtd_tokens_query": tokenizer(batch["query"][item], return_tensors="pt")['input_ids'].shape[1] + tokenizer(batch["positive"][item], return_tensors="pt")['input_ids'].shape[1] + tokenizer(batch["negative"][item], return_tensors="pt")['input_ids'].shape[1]}]))
    return {"counts": pd.concat(qtds_mmarco)}



In [29]:
qtds_mmarco = dataset.map(count_tokens, batched=True, batch_size=2048, num_proc=6, remove_columns=['query', 'positive', 'negative'])


Map (num_proc=6):   0%|          | 0/39780811 [00:00<?, ? examples/s]

In [30]:
qtds_mmarco.save_to_disk('counts_mmarco')

Saving the dataset (0/6 shards):   0%|          | 0/39780811 [00:00<?, ? examples/s]

In [31]:
del dataset

In [3]:
qtds_mmarco = load_from_disk("counts_mmarco")


In [5]:
df = pd.DataFrame(qtds_mmarco['counts'])

In [6]:
del qtds_mmarco

In [12]:
df['qtd_palavras_query'].describe()

count    3.978081e+07
mean     1.332663e+02
std      3.880979e+01
min      1.400000e+01
25%      1.050000e+02
50%      1.260000e+02
75%      1.560000e+02
max      1.788000e+03
Name: qtd_palavras_query, dtype: float64

In [13]:
df['qtd_tokens_query'].describe()

count    3.978081e+07
mean     2.537636e+02
std      8.176989e+01
min      4.400000e+01
25%      1.960000e+02
50%      2.390000e+02
75%      2.960000e+02
max      6.202000e+03
Name: qtd_tokens_query, dtype: float64

In [14]:
qtds_mmarco = []
batch_size=2048

for batch in tqdm(dataset.iter(batch_size=batch_size), total=ds_len//batch_size):
    for item in range(batch_size):
        qtds_mmarco.append({"palavras_query": len(batch["query"][item].split()), "palavras_positive": len(batch["positive"][item].split()), "palavras_negative": len(batch["negative"][item].split()), 
                            "tokens_query": tokenizer(batch["query"][item], return_tensors="pt")['input_ids'].shape[1], "tokens_positive": tokenizer(batch["positive"][item], return_tensors="pt")['input_ids'].shape[1], "tokens_negative": tokenizer(batch["negative"][item], return_tensors="pt")['input_ids'].shape[1]})
        


  0%|          | 34/19424 [00:57<9:05:34,  1.69s/it]


KeyboardInterrupt: 

In [10]:
dataset_collection = load_dataset('unicamp-dl/mmarco', 'collection-portuguese', trust_remote_code=True)
dataset_collection['collection'][100]


Repo card metadata block was not found. Setting CardData to empty.


{'id': 100,
 'text': "AntonÃƒÆ'Ã‚Ân DvorÃƒÆ’Ã‚Â¡k (1841ÃƒÂ ¢ Ã‚â‚¬Ã‚â € œ1904) Antonin Dvorak era filho de um açougueiro, mas não seguia o ofício de seu pai. Enquanto ajudava seu pai em tempo parcial, ele estudou música e se formou na Escola de Órgão de Praga em 1859."}

In [12]:
dataset['train']['query'][0]
#dataset['collection'][0]

'é um pouco de cafeína ok durante a gravidez'

In [15]:
dataset['train']['negative'][0]

'Em geral, é seguro para mulheres grávidas comer chocolate porque estudos demonstraram alguns benefícios de comer chocolate durante a gravidez. No entanto, as mulheres grávidas devem garantir que a ingestão de cafeína seja inferior a 200 mg por dia.'